In [ ]:
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
import seaborn as sns
import fileinput
import warnings
import tensorflow as tf
warnings.filterwarnings('ignore')
import os
import flwr as fl

pd.set_option('display.max_columns', None)
pd.set_option('max_colwidth', None)


In [ ]:
import sys
import sklearn as sk 
import platform

In [ ]:
columns = (['duration'
,'protocol_type'
,'service'
,'flag'
,'src_bytes'
,'dst_bytes'
,'land'
,'wrong_fragment'
,'urgent'
,'hot'
,'num_failed_logins'
,'logged_in'
,'num_compromised'
,'root_shell'
,'su_attempted'
,'num_root'
,'num_file_creations'
,'num_shells'
,'num_access_files'
,'num_outbound_cmds'
,'is_host_login'
,'is_guest_login'
,'count'
,'srv_count'
,'serror_rate'
,'srv_serror_rate'
,'rerror_rate'
,'srv_rerror_rate'
,'same_srv_rate'
,'diff_srv_rate'
,'srv_diff_host_rate'
,'dst_host_count'
,'dst_host_srv_count'
,'dst_host_same_srv_rate'
,'dst_host_diff_srv_rate'
,'dst_host_same_src_port_rate'
,'dst_host_srv_diff_host_rate'
,'dst_host_serror_rate'
,'dst_host_srv_serror_rate'
,'dst_host_rerror_rate'
,'dst_host_srv_rerror_rate'
,'attack'
,'level'])


df_train=pd.read_csv('../data/raw/KDDTrain+.txt',header=None,names=columns)
df_test=pd.read_csv('../data/raw/KDDTest+.txt',header=None,names=columns)


def modify_arff_file(arff_file, modified_arff_file):
    # Open the original ARFF file in read mode
    with open(arff_file, 'r') as original_file:
        # Read the lines of the original ARFF file
        lines = original_file.readlines()

    # Modify the lines to replace " 'icmp'" with 'icmp'
    modified_lines = [line.replace(" 'icmp'", "'icmp'") for line in lines]

    # Write the modified lines to a new ARFF file
    with open(modified_arff_file, 'w') as modified_file:
        modified_file.writelines(modified_lines)
        
test_arff_file = '../data/raw/KDDTest+.arff'
modified_test_arff_file = '../data/raw/KDDTest+_modified.arff'

modify_arff_file(test_arff_file, modified_test_arff_file)

train_arff_file = '../data/raw/KDDTrain+.arff'
modified_train_arff_file = '../data/raw/KDDTrain+_modified.arff'

modify_arff_file(train_arff_file, modified_train_arff_file)

test_data, meta = arff.loadarff('../data/raw/KDDTest+_modified.arff')
train_data, meta = arff.loadarff('../data/raw/KDDTrain+_modified.arff')

df_test2 = pd.DataFrame(test_data)
df_train2 = pd.DataFrame(train_data)

def convert_value(value):
    try:
        return float(value)
    except ValueError:
        return value.decode()

# Iterate over each column of the DataFrame
for column in df_test2.select_dtypes(include='object'):
    # Convert byte strings to either strings or integers
    df_test2[column] = df_test2[column].apply(convert_value)
    df_train2[column] = df_train2[column].apply(convert_value)


In [ ]:

df_train.info()
df_test.info()

df_train.describe()
df_test.describe()

df_test.drop(['num_outbound_cmds'], axis=1, inplace=True)
df_train.drop(['num_outbound_cmds'], axis=1, inplace=True)

def calculate_value_frequencies(df, column):
    value_frequencies = df[column].value_counts()
    total_samples = df.shape[0]

    result_df = pd.DataFrame(columns=['Rank', 'Value', 'Frequency', 'Percentage'])

    for rank, (value, frequency) in enumerate(value_frequencies.items(), start=1):
        percentage = (frequency / total_samples) * 100
        temp_df = pd.DataFrame(data={'Rank': [rank], 'Value': [value], 'Frequency': [frequency], 'Percentage': [percentage]})
        result_df = pd.concat([result_df, temp_df], ignore_index=True)

    return result_df.set_index('Rank')

def plot_histogram(df, column, x=10, y=6):
    value_frequencies = calculate_value_frequencies(df, column)
    print(value_frequencies)
    plt.figure(figsize=(x, y))
    plt.bar(value_frequencies['Value'], value_frequencies['Frequency'])
    plt.xlabel(column)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {column}')
    plt.xticks(rotation=45)
    #plt.show()
    
plot_histogram(df_train, 'attack')

attack_categories = {
    'normal': 'normal',
    'neptune': 'DoS',
    'back': 'DoS',
    'land': 'DoS',
    'pod': 'DoS',
    'smurf': 'DoS',
    'teardrop': 'DoS',
    'mailbomb': 'DoS',
    'apache2': 'DoS',
    'processtable': 'DoS',
    'udpstorm': 'DoS',
    'worm': 'DoS',
    'buffer_overflow': 'U2R',
    'loadmodule': 'U2R',
    'perl': 'U2R',
    'rootkit': 'U2R',
    'ps': 'U2R',
    'sqlattack': 'U2R',
    'xterm': 'U2R',
    'ftp_write': 'R2L',
    'guess_passwd': 'R2L',
    'imap': 'R2L',
    'multihop': 'R2L',
    'phf': 'R2L',
    'spy': 'R2L',
    'warezclient': 'R2L',
    'warezmaster': 'R2L',
    'sendmail': 'R2L',
    'named': 'R2L',
    'snmpgetattack': 'R2L',
    'snmpguess': 'R2L',
    'xlock': 'R2L',
    'xsnoop': 'R2L',
    'httptunnel': 'R2L',
    'ipsweep': 'Probe',
    'nmap': 'Probe',
    'portsweep': 'Probe',
    'satan': 'Probe',
    'mscan': 'Probe',
    'saint': 'Probe'
}

df_train['attack'] = df_train['attack'].map(attack_categories)
df_test['attack'] = df_test['attack'].map(attack_categories)

plot_histogram(df_train, 'attack')
plot_histogram(df_train, 'level')

df_test.drop(['level'], axis=1, inplace=True)
df_train.drop(['level'], axis=1, inplace=True)

plot_histogram(df_train2, 'class')

numeric_categorical_features = df_train.select_dtypes(include=['int', 'float']).columns
df_numeric_categorical = df_train[numeric_categorical_features]

correlation_matrix = df_numeric_categorical.corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

plt.figure(figsize=(35, 35))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', mask=mask)
plt.title('Correlation Heatmap')

#plt.show()

def get_high_correlations(df, threshold=0.5):
    correlation_matrix = df.corr().values

    num_rows, num_cols = correlation_matrix.shape
    data = []

    for i in range(num_rows):
        for j in range(i):
            value = correlation_matrix[i, j]
            value = abs(value)
            if value > threshold:
                data.append([df.columns[i], df.columns[j], value])

    result_df = pd.DataFrame(data, columns=['Column 1', 'Column 2', 'Correlation'])
    result_df = result_df.sort_values(by='Correlation', ascending=False)

    return result_df



get_high_correlations(df_numeric_categorical, threshold=0.85)

def calculate_correlation_sum(df):
    merged_df = pd.DataFrame(columns=['Variable', 'Correlation Sum'])

    merged_df['Variable'] = pd.concat([df['Column 1'], df['Column 2']]).reset_index(drop=True)
    merged_df['Correlation Sum'] = pd.concat([df['Correlation'], df['Correlation']]).reset_index(drop=True)

    correlation_sum = merged_df.groupby('Variable')['Correlation Sum'].sum().sort_values(ascending=False)
    
    correlation_sum_df = pd.DataFrame({'Variable': correlation_sum.index, 'Correlation Sum': correlation_sum.values})
    
    return correlation_sum_df

calculate_correlation_sum(get_high_correlations(df_numeric_categorical, threshold=0.85))

def drop_lowest_correlation(df, threshold=0.5):
    dropped_features = []
  
    while True:
        high_correlations = get_high_correlations(df, threshold)
        
        if high_correlations.empty:
            break
        
        correlation_sums = calculate_correlation_sum(high_correlations.tail(1))
        lowest_correlation = correlation_sums.iloc[0]['Variable']
        
        df = df.drop([lowest_correlation], axis=1)
        
        dropped_features.append(lowest_correlation)
        print(f"Dropped feature: {lowest_correlation}")
    
    return df, dropped_features



print(df_numeric_categorical.shape)
df_numeric_categorical, dropped_features = drop_lowest_correlation(df_numeric_categorical, threshold=0.85)
print(df_numeric_categorical.shape)
get_high_correlations(df_numeric_categorical, threshold=0.70)

print("Features dropped based on high correlation: \n")

print(dropped_features)
print(df_test.shape)
df_test.drop(dropped_features, axis=1, inplace=True)
print(df_test.shape)
print(df_train.shape)
df_train.drop(dropped_features, axis=1, inplace=True)
print(df_train.shape)

def plot_histograms(df, feature, x=10, y=10, title=True):
    unique_features = df[feature].unique()
    value_frequencies = df[feature].value_counts()
    sorted_features = value_frequencies.index.tolist()
    num_plots = len(unique_features)
    num_cols = 2
    num_rows = (num_plots + num_cols - 1) // num_cols
    
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(x, y))
    
    if title:
        fig.suptitle(f"Distribution of attack for each {feature}", fontsize=14, ha='center')
    
    for i, feat in enumerate(sorted_features):
        feat_df = df[df[feature] == feat]
        unique_attacks = feat_df['attack'].unique()
        
        row = i // num_cols
        col = i % num_cols
        ax = axes[row, col]
        
        sns.histplot(feat_df['attack'], bins=len(unique_attacks), ax=ax)
        ax.set_xlabel('Attack')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{feature}: {feat}')
    
    # Remove any empty subplots
    if num_plots < num_rows * num_cols:
        empty_plots = num_rows * num_cols - num_plots
        for i in range(num_plots, num_plots + empty_plots):
            fig.delaxes(axes[i // num_cols, i % num_cols])
    
    plt.tight_layout()
    #plt.show()

df_train2 = df_train2[df_train2['protocol_type']=='tcp']
df_test2 = df_test2[df_test2['protocol_type']=='tcp']

df_train2.drop(columns='protocol_type', axis=1, inplace=True)
df_test2.drop(columns='protocol_type', axis=1, inplace=True)

df_train2.keys()


In [ ]:

print(calculate_value_frequencies(df_train2, 'urgent')) #Dropping this feature because the majority of the values (0.99) takes value 0
print(calculate_value_frequencies(df_train2, 'wrong_fragment')) #Dropping this feature because the majority of the values (0.99) takes value 0
print(calculate_value_frequencies(df_train2, 'hot')) #97.87% of values are 0
print(calculate_value_frequencies(df_train2, 'duration')) #92.04% of values are 0
print(calculate_value_frequencies(df_train2, 'src_bytes')) #Other than the presence of the outliers, there isn't a presence of a value that takes over 90% of the dataset
print(calculate_value_frequencies(df_train2, 'dst_bytes')) #Other than the presence of the outliers, there isn't a presence of a value that takes over 90% of the dataset
print(calculate_value_frequencies(df_train2, 'num_failed_logins')) # the majority of the values (99.9%) takes value 0
print(calculate_value_frequencies(df_train2, 'num_compromised')) #98.97% of the values are 0
print(calculate_value_frequencies(df_train2, 'num_file_creations')) #the majority of the values (99.77%) takes value 0
print(calculate_value_frequencies(df_train2, 'num_shells')) # the majority of the values (99.96%) takes value 0
print(calculate_value_frequencies(df_train2, 'num_access_files')) # the majority of the values (99.7%) takes value 0
print(calculate_value_frequencies(df_train2, 'count')) #Other than the presence of the outliers, there isn't a presence of a value that takes over 90% of the dataset
print(calculate_value_frequencies(df_train2, 'srv_count')) #Other than the presence of the outliers, there isn't a presence of a value that takes over 90% of the dataset

df_train2.drop(columns=['service','flag','num_outbound_cmds','num_root','src_bytes'
    ,'dst_bytes'
    ,'urgent'
    ,'num_failed_logins'
    ,'num_compromised'
    ,'root_shell'
    ,'su_attempted'
    ,'num_root'
    ,'num_file_creations'
    ,'num_shells'
    ,'num_access_files'
    ,'is_host_login'
    ,'is_guest_login'
    ,'wrong_fragment'], axis=1, inplace=True)
df_test2.drop(columns=['service','flag','num_outbound_cmds','num_root','src_bytes'
    ,'dst_bytes'
    ,'urgent'
    ,'num_failed_logins'
    ,'num_compromised'
    ,'root_shell'
    ,'su_attempted'
    ,'num_root'
    ,'num_file_creations'
    ,'num_shells'
    ,'num_access_files'
    ,'is_host_login'
    ,'is_guest_login'
    ,'wrong_fragment'],axis=1, inplace=True)

print("Features dropped based on value frequencies: ", columns)

print(df_test2.head())


In [ ]:
#df_test2.to_csv('test.csv', index=False)

In [ ]:

categorical_columns = ['protocol_type', 'service', 'flag']

df_train_encoded = pd.get_dummies(df_train, columns=categorical_columns)
df_test_encoded = pd.get_dummies(df_test, columns=categorical_columns)

df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, axis=1, join='inner')

from sklearn.preprocessing import RobustScaler

X = df_train_encoded.drop('attack', axis=1)
y = df_train_encoded['attack']

scaler = RobustScaler()
df_train_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

df_train_scaled['attack'] = y

df_train2[['land', 'logged_in']] = df_train2[['land', 'logged_in']].astype(float)
df_test2[['land', 'logged_in']] = df_test2[['land', 'logged_in']].astype(float)

df_train2['class'] = (df_train2['class'] != 'normal').astype(int)
df_test2['class'] = (df_test2['class'] != 'normal').astype(int)

df_train2 = df_train2.reset_index().drop(columns='index')
df_test2 = df_test2.reset_index().drop(columns='index')

X_train = df_train2.drop('class', axis=1)
y_train = df_train2['class']

X_test = df_test2.drop('class', axis=1)
y_test = df_test2['class']
d_raw_train = df_train2.copy()
d_raw_test = df_test2.copy()

d_raw_train_normal = d_raw_train[d_raw_train['class'] == 0]
X_train_normal = d_raw_train_normal.drop('class', axis=1)
y_train_normal = 0



In [ ]:

# df_train.drop(columns=['service','flag','num_outbound_cmds','num_root','src_bytes'
#     ,'dst_bytes'
#     ,'urgent'
#     ,'num_failed_logins'
#     ,'num_compromised'
#     ,'root_shell'
#     ,'su_attempted'
#     ,'num_root'
#     ,'num_file_creations'
#     ,'num_shells'
#     ,'num_access_files'
#     ,'is_host_login'
#     ,'is_guest_login'
#     ,'wrong_fragment'], axis=1, inplace=True)

# df_train[['land', 'logged_in']] = df_train[['land', 'logged_in']].astype(float)
# df_train['class'] = (df_train['class'] != 'normal').astype(int)
# df_train = df_train.reset_index().drop(columns='index')

# X_train = df_train2.drop('class', axis=1)
# y_train = df_train2['class']



In [ ]:
df_train2.shape

In [ ]:

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

d_norm_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
d_norm_train['class'] = y_train

d_norm_test = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)
d_norm_test['class'] = y_test

d_norm_train_normal = pd.DataFrame(scaler.fit_transform(X_train_normal), columns=X_train_normal.columns)
d_norm_train_normal['class'] = y_train_normal
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.mixture import GaussianMixture

def Best_n_components(df, params_grid,model):
    #pca = PCA()
    gsv =GridSearchCV(model, params_grid)
    gsv.fit(df)
    return gsv.best_params_




In [ ]:
d_norm_train['class'].value_counts()

In [ ]:
# d_norm_train.to_csv('d_norm_train.csv', index=False)

In [ ]:


def PCA_Transformation(df, dim=None):
    if dim:
        pca = PCA(dim)
        data_transformed = pca.fit_transform(df)
        return pd.DataFrame(data_transformed)
    pca = PCA()
    params = {'n_components': np.arange(1, 24)}
    n = Best_n_components(df, params,pca)['n_components']
    pca = PCA(n)
    data_transformed = pca.fit_transform(df)
    return pd.DataFrame(data_transformed), n

d_raw_pca_train,n = PCA_Transformation(d_raw_train)

d_raw_pca_test = PCA_Transformation(d_raw_test,n)

d_raw_pca_train_normal = PCA_Transformation(d_raw_train_normal,n)

d_norm_pca_train,n = PCA_Transformation(d_norm_train)
print(n)
d_norm_pca_test = PCA_Transformation(d_norm_test,n)
print(n)
d_norm_pca_train_normal = PCA_Transformation(d_raw_train_normal,n)
print(n)


In [ ]:

from tqdm import tqdm
from scipy.stats import norm

def FGMPM(original_data, data):
    #gmm = GaussianMixture()
    #params = {'n_components': np.arange(1, 6)}
    #n = Best_n_components(original_data, params,gmm)['n_components']
    gmm = GaussianMixture(n_components=20,random_state=0)
    gmm.fit(original_data)
    fgmpm = []
    for i in tqdm(range(len(data))):
        row_values = data.iloc[i,:]
        row_probs = []
        memberships = gmm.predict([row_values])
        means = gmm.means_[memberships[0]]
        covariances = np.sqrt(np.diag(gmm.covariances_[memberships[0]]))

        z_scores = (row_values - means) / covariances

        probs = (1 - norm.cdf(z_scores)) 
        row_probs.extend(probs)

        fgmpm.append(row_probs)
    return pd.DataFrame(fgmpm)
#Works but very long

def vote_FGMPM(original_data, threshold):
   
    abnormal_list = (original_data > threshold).sum(axis=1)       
    return abnormal_list

d_raw_probs_train = FGMPM(d_raw_train,d_raw_train)
d_raw_probs_test = FGMPM(d_raw_train,d_raw_test)
#normal

d_raw_probs_train_normal = FGMPM(d_raw_train_normal,d_raw_train_normal)

d_norm_probs_train = FGMPM(d_norm_train,d_norm_train)
d_norm_probs_test = FGMPM(d_norm_train,d_norm_test)
#normal
d_norm_probs_train_normal = FGMPM(d_norm_train_normal,d_norm_train_normal)

d_raw_pca_probs_train = FGMPM(d_raw_pca_train,d_raw_pca_train)
d_raw_pca_probs_test = FGMPM(d_raw_pca_train,d_raw_pca_test)
#normal
d_raw_pca_probs_train_normal = FGMPM(d_raw_pca_train_normal,d_raw_pca_train_normal)

d_norm_pca_probs_train = FGMPM(d_norm_pca_train,d_norm_pca_train)
d_norm_pca_probs_test = FGMPM(d_norm_pca_train,d_norm_pca_test)
#normal
d_norm_pca_probs_train_normal = FGMPM(d_norm_pca_train_normal,d_norm_pca_train_normal)



In [ ]:

print("The encoded data: \n", df_train_encoded.head())
print("\n")
print("The scaled data is: \n", df_train_scaled.head())
print("\n")
print("Normalised some more: \n", d_norm_train_normal.head())
print("\n")
print("d_raw_train_normal: \n", d_raw_train_normal.head())
#d_raw_train_normal.to_csv("d_raw_train_normal", index= False)
print("d_norm_train_normal: \n", d_norm_train_normal.head())
d_norm_train_normal.to_csv("d_norm_train_normal", index= False)
d_norm_test.to_csv("d_norm_test", index = False)


In [ ]:
"""
d_raw_probs_train.to_csv('d_raw_probs_train.csv', index=False)
d_raw_probs_test.to_csv('d_raw_probs_test.csv', index=False)
d_raw_probs_train_normal.to_csv('d_raw_probs_train_normal.csv', index=False)
d_norm_probs_train.to_csv('d_norm_probs_train.csv', index=False)
d_norm_probs_test.to_csv('d_norm_probs_test.csv', index=False)
d_norm_probs_train_normal.to_csv('d_norm_probs_train_normal.csv', index=False)
d_raw_pca_probs_train.to_csv('d_raw_pca_probs_train.csv', index=False)
d_raw_pca_probs_test.to_csv('d_raw_pca_probs_test.csv', index=False)
d_raw_pca_probs_train_normal.to_csv('d_raw_pca_probs_train_normal.csv', index=False)
d_norm_pca_probs_train.to_csv('d_norm_pca_probs_train.csv', index=False)
d_norm_pca_probs_test.to_csv('d_norm_pca_probs_test.csv', index=False)
d_norm_pca_probs_train_normal.to_csv('d_norm_pca_probs_train_normal.csv', index=False)

"""

In [ ]:
import pandas as pd

d_norm_train_normal = pd.read_csv('d_norm_train_normal.csv')
print(d_norm_train_normal.shape)
d_norm_test['class'].value_counts()


In [ ]:
# import numpy as np
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Reshape, Flatten
# from tensorflow.keras.optimizers.legacy import Adam

# def build_generator(latent_dim, data_dim):
#     model = Sequential([
#         Dense(256, input_dim=latent_dim),
#         LeakyReLU(alpha=0.2),
#         BatchNormalization(momentum=0.8),
#         Dense(512),
#         LeakyReLU(alpha=0.2),
#         BatchNormalization(momentum=0.8),
#         Dense(1024),
#         LeakyReLU(alpha=0.2),
#         BatchNormalization(momentum=0.8),
#         Dense(data_dim, activation='sigmoid')  # Adjusted to match the number of features in your dataset
#     ])
#     return model

# def build_discriminator(input_shape):
#     model = Sequential([
#         Flatten(input_shape=input_shape),
#         Dense(512),
#         LeakyReLU(alpha=0.2),
#         Dense(256),
#         LeakyReLU(alpha=0.2),
#         Dense(1, activation='sigmoid')
#     ])
#     return model

# # Dimensions specific to your dataset
# latent_dim = 100  # Dimensionality of the input noise to the generator
# data_dim = 24  # Number of features in your dataset
# input_shape = (data_dim, )  # Adjusted for the shape of your dataset

# # Build the models
# generator = build_generator(latent_dim, data_dim)
# discriminator = build_discriminator(input_shape)

# # Compile the discriminator
# discriminator.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5), metrics=['accuracy'])

# # Output: Model summaries to verify the structure
# generator.summary()
# discriminator.summary()


In [ ]:

# import numpy as np
# from tensorflow.keras.models import Model
# from tensorflow.keras.layers import Input
# from tensorflow.keras.optimizers.legacy import Adam
# from tqdm import tqdm  # Import tqdm for progress bar

# # Assuming latent_dim is the dimensionality of your generator's input
# latent_dim = 100

# # Combine generator and discriminator for the combined model
# discriminator.trainable = False  # Freeze discriminator during generator training
# gan_input = Input(shape=(latent_dim,))
# gan_output = discriminator(generator(gan_input))
# gan_model = Model(gan_input, gan_output)

# # Compile the combined model
# gan_model.compile(loss='binary_crossentropy', optimizer=Adam(0.0002, 0.5))

# # Training parameters
# epochs = 10000
# batch_size = 32
# sample_interval = 1000  # Interval to save some generated samples for monitoring

# # Load and prepare your dataset
# X_train = d_norm_train_normal  # Assuming this is your preprocessed training data

# # Training loop with tqdm
# for epoch in tqdm(range(epochs), desc='Training Progress'):

#     # ---------------------
#     #  Train Discriminator
#     # ---------------------

#     # Select a random batch of real data
#     idx = np.random.randint(0, X_train.shape[0], batch_size)
#     real_data = X_train.iloc[idx]

#     # Generate a batch of fake data
#     noise = np.random.normal(0, 1, (batch_size, latent_dim))
#     fake_data = generator.predict(noise)

#     # Labels for real and fake data
#     real_y = np.ones((batch_size, 1))
#     fake_y = np.zeros((batch_size, 1))

#     # Train the discriminator
#     d_loss_real = discriminator.train_on_batch(real_data, real_y)
#     d_loss_fake = discriminator.train_on_batch(fake_data, fake_y)
#     d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

#     # ---------------------
#     #  Train Generator
#     # ---------------------

#     # Train the generator (to have the discriminator label samples as real)
#     noise = np.random.normal(0, 1, (batch_size, latent_dim))
#     g_loss = gan_model.train_on_batch(noise, np.ones((batch_size, 1)))

#     # If at save interval => save generated data samples
#     if epoch % sample_interval == 0:
#         # Save model checkpoints or generated samples for inspection
#         print(f"{epoch} [D loss: {d_loss[0]}, acc.: {100*d_loss[1]}] [G loss: {g_loss}]")
#         # Optionally, implement saving of generated samples here

# # Note: This code will display a progress bar using tqdm, which provides a visual indication of training progress.
